# DOAR E1-A2 GPU Runner

Launcher only -- all scientific logic lives in `experiments/E1_visual_representation/scripts/` in the repository. This notebook clones the repo, verifies the frozen T0 split, and calls `run_e1a_all.py`. It never duplicates any E1 logic locally.

**Frozen baseline (never changed by this notebook):** checkpoint `checkpoint/t0-clean-split-v1`, T0 manifest SHA-256 `4631ce8bddde64755ba44758827310703f1b332bcb28f72b55f22b3b19b92c9d`, Train=2599 / Valid=284 / Test=512 (locked, never evaluated here).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Variables -- edit these for your environment

In [ ]:
GITHUB_REPO = "https://github.com/HHemaly/DOAR.git"
BRANCH = "feature/doar-phase2c-annotation-expansion"
DATASET_ROOT = "/content/drive/MyDrive/DOAR/data/Combined_Drawing"
OUTPUT_ROOT = "/content/drive/MyDrive/DOAR/gpu_runs/E1/E1A"
REPO_DIR = "/content/DOAR"

## 3. Clone/pull the repository

In [ ]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch "$BRANCH" "$GITHUB_REPO" "$REPO_DIR"
else:
    !cd "$REPO_DIR" && git checkout "$BRANCH" && git pull origin "$BRANCH"
%cd $REPO_DIR

## 4. Install E1 requirements (does not touch Colab's own CUDA PyTorch build)

In [ ]:
!pip install -q -r experiments/E1_visual_representation/requirements-e1.txt --upgrade-strategy only-if-needed

## 5. Verify CUDA/GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError("No CUDA device detected -- select Runtime > Change runtime type > GPU.")

## 6. Verify dataset + frozen T0 (never regenerates or edits the split)

In [ ]:
!python experiments/E1_visual_representation/scripts/prepare_e1_dataset.py \
  --dataset-root "$DATASET_ROOT" --verify-only

## 7. Execute run_e1a_all.py

In [ ]:
!python experiments/E1_visual_representation/scripts/run_e1a_all.py \
  --dataset-root "$DATASET_ROOT" \
  --output-root "$OUTPUT_ROOT" \
  --device cuda --seed 42 --max-epochs 30 --patience 5 --resume

## 8. Display final E1-A table and figure paths

In [ ]:
import json, glob
print("RUN_MANIFEST.json:")
print(json.dumps(json.load(open(f"{OUTPUT_ROOT}/RUN_MANIFEST.json")), indent=2))
print("\nTables (repo, committable):")
for p in sorted(glob.glob("experiments/E1_visual_representation/tables/*.csv")):
    print(" ", p)
print("\nFigures (repo, committable):")
for p in sorted(glob.glob("experiments/E1_visual_representation/figures/*.png")):
    print(" ", p)
print(f"\nLarge artifacts (checkpoints/embeddings) remain in Drive only: {OUTPUT_ROOT}")